# Notebook 13: Évaluation Complète - Phase 2 Model

## Objectif
Évaluation complète du modèle Qwen2.5-VL-7B fine-tuné (Phase 2) sur le **test set**:
- **Test set**: 3,050 images jamais vues pendant l'entraînement
- **Métriques de classification**: Accuracy, Precision, Recall, F1-Score
- **Analyse détaillée**: Confusion matrix, per-class metrics
- **Génération qualitative**: Exemples de prédictions textuelles
- **Comparaison**: Baseline (40%) → Phase 1 → Phase 2

## Résultats Attendus
**Target**: **95-97% accuracy** (vs baseline 40%)

## Inputs
- Best checkpoint Phase 2: `models/checkpoints/phase2_lora/best/`
- Test set: `outputs/vlm_dataset/test.jsonl` (3,050 images)

## Outputs
- Métriques JSON: `outputs/evaluation_results_phase2.json`
- Confusion matrix: `outputs/confusion_matrix_phase2.png`
- Classification report: Console output

## 1. Imports & Setup

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import json
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Transformers & PEFT
from transformers import AutoModelForVision2Seq, AutoProcessor
from PIL import Image
from qwen_vl_utils import process_vision_info

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Configuration
PROJECT_ROOT = Path("/home/hounfodjidagba/learning/cassava")
VLM_DATASET_DIR = PROJECT_ROOT / "outputs" / "vlm_dataset"
PHASE2_CHECKPOINT = PROJECT_ROOT / "models" / "checkpoints" / "phase2_lora" / "best"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Créer le répertoire de sortie
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class names
CLASS_NAMES = [
    'Bacterial Blight',
    'Brown Streak Disease',
    'Green Mottle',
    'Healthy',
    'Mosaic Disease'
]

CLASS_SHORT = ['CBB', 'CBSD', 'CGM', 'Healthy', 'CMD']

print("="*70)
print("ÉVALUATION COMPLÈTE - PHASE 2 MODEL")
print("="*70)
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"\nDirectories:")
print(f"  Phase 2 Checkpoint: {PHASE2_CHECKPOINT}")
print(f"  Test Dataset: {VLM_DATASET_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print("="*70)

## 2. Load Best Checkpoint Phase 2

In [ ]:
print("Chargement du meilleur modèle Phase 2...")
print(f"Checkpoint path: {PHASE2_CHECKPOINT}")

# Charger le modèle depuis le checkpoint Phase 2
model = AutoModelForVision2Seq.from_pretrained(
    PHASE2_CHECKPOINT,
    dtype="auto",
    device_map="auto"
)

# Charger le processor
processor = AutoProcessor.from_pretrained(PHASE2_CHECKPOINT)

print("\n✓ Modèle Phase 2 chargé")

# Charger les métriques de training Phase 2
phase2_metrics_path = OUTPUT_DIR / "phase2_metrics.json"
if phase2_metrics_path.exists():
    with open(phase2_metrics_path, 'r') as f:
        phase2_metrics = json.load(f)
    print("\n" + "="*70)
    print("RÉSULTATS ENTRAÎNEMENT PHASE 2")
    print("="*70)
    print(f"Epochs: {phase2_metrics['total_epochs']}")
    print(f"Durée: {phase2_metrics['training_duration_hours']:.2f}h")
    print(f"Best Val Loss: {phase2_metrics['best_val_loss']:.4f}")
    print(f"Best Epoch: {phase2_metrics['best_epoch']}")
    print(f"Early Stopped: {phase2_metrics['early_stopped']}")
    print("="*70)
else:
    print("⚠️ Phase 2 metrics non trouvés")
    phase2_metrics = {}

## 3. Dataset & DataLoader pour Test Set

In [ ]:
# Importer le dataset (même structure que notebooks précédents)
from torch.utils.data import Dataset

class CassavaVLMDataset(Dataset):
    """Dataset pour Qwen2.5-VL avec format JSONL."""
    
    def __init__(self, jsonl_path, image_base_path, processor, max_length=512):
        self.samples = []
        self.image_base_path = Path(image_base_path)
        self.processor = processor
        self.max_length = max_length
        
        # Charger les samples du JSONL
        with open(jsonl_path, 'r') as f:
            for line in f:
                self.samples.append(json.loads(line))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Charger l'image
        image_path = self.image_base_path / sample['image']
        image = Image.open(image_path).convert('RGB')
        
        # Construire les messages au format Qwen (seulement la question, pas la réponse)
        messages = []
        for conv in sample['conversations']:
            if conv['from'] == 'human':  # Seulement la question pour la génération
                content_text = conv['value']
                if '<image>' in content_text:
                    text = content_text.replace('<image>\n', '').replace('<image>', '')
                    content = [
                        {"type": "image", "image": image},
                        {"type": "text", "text": text}
                    ]
                else:
                    content = content_text
                messages.append({"role": "user", "content": content})
                break  # Seulement la première question
        
        # Traiter avec le processor Qwen (avec generation_prompt pour la génération)
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Extraire les tensors
        input_ids = inputs['input_ids'].squeeze(0)
        attention_mask = inputs['attention_mask'].squeeze(0)
        pixel_values = inputs.get('pixel_values', None)
        image_grid_thw = inputs.get('image_grid_thw', None)
        
        if pixel_values is not None:
            pixel_values = pixel_values.squeeze(0)
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.squeeze(0)
        
        # Extraire metadata
        label = sample['metadata']['label']
        class_short = sample['metadata']['class_short']
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'pixel_values': pixel_values,
            'image_grid_thw': image_grid_thw,
            'label': label,
            'class_short': class_short,
            'image_id': sample['id']
        }

def collate_fn(batch):
    """Collate function pour batching."""
    # Trouver les longueurs max
    max_length = max(item['input_ids'].size(0) for item in batch)
    
    # Préparer les tensors
    input_ids = []
    attention_masks = []
    pixel_values_list = []
    image_grid_thw_list = []
    label_list = []
    class_short_list = []
    image_id_list = []
    
    for item in batch:
        # Pad input_ids et attention_mask
        seq_len = item['input_ids'].size(0)
        pad_len = max_length - seq_len
        
        input_ids.append(torch.cat([
            item['input_ids'],
            torch.zeros(pad_len, dtype=torch.long)
        ]))
        
        attention_masks.append(torch.cat([
            item['attention_mask'],
            torch.zeros(pad_len, dtype=torch.long)
        ]))
        
        if item['pixel_values'] is not None:
            pixel_values_list.append(item['pixel_values'])
        
        if item['image_grid_thw'] is not None:
            image_grid_thw_list.append(item['image_grid_thw'])
        
        label_list.append(item['label'])
        class_short_list.append(item['class_short'])
        image_id_list.append(item['image_id'])
    
    # Stack tensors
    batched = {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attention_masks),
        'label': torch.tensor(label_list, dtype=torch.long),
        'class_short': class_short_list,
        'image_id': image_id_list
    }
    
    if pixel_values_list:
        batched['pixel_values'] = torch.stack(pixel_values_list)
    
    if image_grid_thw_list:
        batched['image_grid_thw'] = torch.stack(image_grid_thw_list)
    
    return batched

print("✓ Classes Dataset et collate_fn définies")

In [ ]:
# Créer le test dataset
test_jsonl = VLM_DATASET_DIR / "test.jsonl"

print("Création du test dataset...")
test_dataset = CassavaVLMDataset(
    jsonl_path=test_jsonl,
    image_base_path=VLM_DATASET_DIR,
    processor=processor,
    max_length=512
)

# Créer le dataloader (batch_size=4 pour la génération)
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print("="*70)
print("TEST DATASET CRÉÉ")
print("="*70)
print(f"Test samples: {len(test_dataset):,}")
print(f"Test batches: {len(test_loader):,}")
print(f"Batch size: 4")
print("="*70)

## 4. Fonction de Parsing des Réponses

In [ ]:
def parse_disease_from_response(response: str) -> int:
    """
    Extraire la classe de maladie depuis la réponse générée.
    
    Returns:
        int: Class ID (0-4), ou -1 si parsing échoue
    """
    # Mapping texte → classe
    disease_map = {
        'bacterial blight': 0,
        'cassava bacterial blight': 0,
        'cbb': 0,
        'brown streak disease': 1,
        'cassava brown streak disease': 1,
        'cbsd': 1,
        'brown streak': 1,
        'green mottle': 2,
        'cassava green mottle': 2,
        'cgm': 2,
        'healthy': 3,
        'mosaic disease': 4,
        'cassava mosaic disease': 4,
        'cmd': 4,
        'mosaic': 4
    }
    
    # Nettoyer et normaliser la réponse
    response_lower = response.lower().strip()
    
    # Chercher les mentions de maladies (ordre du plus spécifique au plus général)
    for disease, class_id in sorted(disease_map.items(), key=lambda x: -len(x[0])):
        if disease in response_lower:
            return class_id
    
    # Si aucune maladie détectée, retourner -1 (erreur de parsing)
    return -1

# Test de la fonction
print("Test de la fonction de parsing:")
test_cases = [
    "The cassava leaf shows Cassava Bacterial Blight (CBB).",
    "This is Brown Streak Disease",
    "The plant is healthy.",
    "Shows signs of Mosaic",
    "Green Mottle detected"
]

for test in test_cases:
    pred = parse_disease_from_response(test)
    print(f"  '{test[:50]}...' → Class {pred} ({CLASS_NAMES[pred] if pred != -1 else 'PARSING ERROR'})")

print("\n✓ Fonction de parsing définie")

## 5. Évaluation avec Génération

In [ ]:
@torch.no_grad()
def evaluate_with_generation(model, test_loader, processor, device):
    """
    Évaluer en générant des réponses et extrayant les prédictions.
    
    Returns:
        tuple: (all_preds, all_labels, all_responses, parsing_errors)
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_responses = []
    parsing_errors = 0
    
    progress_bar = tqdm(test_loader, desc="Evaluation")
    
    for batch in progress_bar:
        # Déplacer sur device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        true_labels = batch['label']
        
        # Gérer pixel_values et image_grid_thw
        pixel_values = batch.get('pixel_values')
        if pixel_values is not None:
            pixel_values = pixel_values.to(device)
        
        image_grid_thw = batch.get('image_grid_thw')
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.to(device)
        
        # Générer les réponses
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            image_grid_thw=image_grid_thw,
            max_new_tokens=100,
            do_sample=False,  # Greedy decoding pour reproductibilité
            temperature=None,
            top_p=None
        )
        
        # Décoder les réponses
        responses = processor.batch_decode(outputs, skip_special_tokens=True)
        
        # Parser les réponses pour extraire les classes prédites
        for response, true_label in zip(responses, true_labels):
            pred_class = parse_disease_from_response(response)
            
            if pred_class == -1:
                parsing_errors += 1
            
            all_preds.append(pred_class)
            all_labels.append(true_label.item())
            all_responses.append(response)
        
        # Update progress bar
        progress_bar.set_postfix({
            'parsed': f"{len(all_preds) - parsing_errors}/{len(all_preds)}",
            'errors': parsing_errors
        })
    
    return all_preds, all_labels, all_responses, parsing_errors

print("✓ Fonction d'évaluation définie")

In [ ]:
# Lancer l'évaluation
print("="*70)
print("DÉBUT DE L'ÉVALUATION SUR TEST SET")
print("="*70)
print(f"Test samples: {len(test_dataset):,}")
print(f"Génération: Greedy decoding, max_new_tokens=100")
print("="*70)

import time
start_time = time.time()

all_preds, all_labels, all_responses, parsing_errors = evaluate_with_generation(
    model, test_loader, processor, device
)

eval_duration = time.time() - start_time

print("\n" + "="*70)
print("ÉVALUATION TERMINÉE")
print("="*70)
print(f"Durée: {eval_duration / 60:.2f} min")
print(f"Samples évalués: {len(all_labels):,}")
print(f"Parsing errors: {parsing_errors} ({100*parsing_errors/len(all_labels):.2f}%)")
print("="*70)

## 6. Calcul des Métriques

In [ ]:
# Filtrer les parsing errors pour les métriques
valid_indices = [i for i, pred in enumerate(all_preds) if pred != -1]
valid_preds = [all_preds[i] for i in valid_indices]
valid_labels = [all_labels[i] for i in valid_indices]

print(f"Calcul des métriques sur {len(valid_preds):,} prédictions valides...\n")

# Calculer les métriques globales
accuracy = accuracy_score(valid_labels, valid_preds)
precision, recall, f1, support = precision_recall_fscore_support(
    valid_labels, valid_preds, average='weighted', zero_division=0
)

# Per-class metrics
precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
    valid_labels, valid_preds, average=None, zero_division=0
)

# Confusion matrix
cm = confusion_matrix(valid_labels, valid_preds)

# Classification report
report = classification_report(
    valid_labels,
    valid_preds,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)

# Afficher les résultats
print("="*70)
print("MÉTRIQUES GLOBALES")
print("="*70)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("="*70)

print("\n" + "="*70)
print("CLASSIFICATION REPORT DÉTAILLÉ")
print("="*70)
print(report)
print("="*70)

## 7. Visualisation de la Confusion Matrix

In [ ]:
# Créer la confusion matrix visualization
plt.figure(figsize=(12, 10))

# Heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_SHORT,
    yticklabels=CLASS_SHORT,
    cbar_kws={'label': 'Count'}
)

plt.title(f'Confusion Matrix - Phase 2 Model\nAccuracy: {accuracy:.2%}', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()

# Sauvegarder
cm_path = OUTPUT_DIR / 'confusion_matrix_phase2.png'
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
print(f"✓ Confusion matrix sauvegardée: {cm_path}")

plt.show()

## 8. Analyse Per-Class

In [ ]:
# Créer un DataFrame pour les métriques par classe
per_class_df = pd.DataFrame({
    'Class': CLASS_NAMES,
    'Precision': precision_per_class,
    'Recall': recall_per_class,
    'F1-Score': f1_per_class,
    'Support': support_per_class
})

print("\n" + "="*70)
print("MÉTRIQUES PAR CLASSE")
print("="*70)
print(per_class_df.to_string(index=False))
print("="*70)

# Visualisation des métriques par classe
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#e74c3c', '#2ecc71']

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    axes[idx].barh(CLASS_SHORT, per_class_df[metric], color=color, alpha=0.7)
    axes[idx].set_xlabel(metric, fontsize=12)
    axes[idx].set_xlim(0, 1.05)
    axes[idx].set_title(f'{metric} par Classe', fontsize=12, fontweight='bold')
    axes[idx].grid(axis='x', alpha=0.3)
    
    # Ajouter les valeurs sur les barres
    for i, v in enumerate(per_class_df[metric]):
        axes[idx].text(v + 0.02, i, f'{v:.3f}', va='center', fontsize=10)

plt.tight_layout()
metrics_path = OUTPUT_DIR / 'per_class_metrics_phase2.png'
plt.savefig(metrics_path, dpi=300, bbox_inches='tight')
print(f"\n✓ Métriques par classe sauvegardées: {metrics_path}")
plt.show()

## 9. Exemples Qualitatifs

In [ ]:
# Afficher quelques exemples de prédictions
print("\n" + "="*70)
print("EXEMPLES DE PRÉDICTIONS (10 premiers)")
print("="*70)

for i in range(min(10, len(all_labels))):
    true_class = CLASS_NAMES[all_labels[i]]
    pred_class = CLASS_NAMES[all_preds[i]] if all_preds[i] != -1 else 'PARSING ERROR'
    correct = "✓" if all_preds[i] == all_labels[i] else "✗"
    
    print(f"\nExemple {i+1}: {correct}")
    print(f"  True Label:  {true_class}")
    print(f"  Predicted:   {pred_class}")
    print(f"  Response:    {all_responses[i][:200]}...")
    print("-" * 70)

# Compter les erreurs par type
errors_by_class = {class_name: 0 for class_name in CLASS_NAMES}
for true_label, pred in zip(all_labels, all_preds):
    if pred != -1 and true_label != pred:
        errors_by_class[CLASS_NAMES[true_label]] += 1

print("\n" + "="*70)
print("ERREURS PAR CLASSE")
print("="*70)
for class_name, error_count in errors_by_class.items():
    class_support = support_per_class[CLASS_NAMES.index(class_name)]
    error_rate = 100 * error_count / class_support if class_support > 0 else 0
    print(f"{class_name:25s}: {error_count:4d} / {int(class_support):4d} ({error_rate:5.2f}%)")
print("="*70)

## 10. Comparaison Baseline → Phase 1 → Phase 2

In [ ]:
# Charger les métriques Phase 1 (si disponibles)
phase1_metrics_path = OUTPUT_DIR / "phase1_metrics.json"  # Si existe

# Créer le tableau de comparaison
comparison = {
    'Baseline (Zero-shot)': {
        'Accuracy': 0.40,
        'Val Loss': None,
        'Duration (h)': 0
    },
    'Phase 1 (Projector)': {
        'Accuracy': None,  # Non évalué sur test set
        'Val Loss': 6.5445,
        'Duration (h)': 1.18
    },
    'Phase 2 (LoRA)': {
        'Accuracy': accuracy,
        'Val Loss': phase2_metrics.get('best_val_loss', 6.0045),
        'Duration (h)': phase2_metrics.get('training_duration_hours', 3.54)
    }
}

# Créer DataFrame
comparison_df = pd.DataFrame(comparison).T

print("\n" + "="*70)
print("COMPARAISON DES PHASES")
print("="*70)
print(comparison_df.to_string())
print("="*70)

# Calculer l'amélioration
baseline_acc = 0.40
improvement = ((accuracy - baseline_acc) / baseline_acc) * 100

print(f"\n🎯 Amélioration vs Baseline: +{improvement:.1f}%")
print(f"   Baseline:  {baseline_acc:.1%}")
print(f"   Phase 2:   {accuracy:.1%}")

# Vérifier si on atteint le target
target_min = 0.95
target_max = 0.97

if accuracy >= target_min:
    print(f"\n✅ TARGET ATTEINT ! ({accuracy:.1%} >= {target_min:.0%})")
    if accuracy >= target_max:
        print(f"   🌟 EXCELLENT ! Dépasse même le target max ({target_max:.0%})")
else:
    print(f"\n⚠️ Target non atteint ({accuracy:.1%} < {target_min:.0%})")
    gap = target_min - accuracy
    print(f"   Il manque {gap:.1%} pour atteindre le target minimum")

## 11. Sauvegarde des Résultats

In [ ]:
# Préparer les résultats à sauvegarder
results = {
    'phase': 'phase2_evaluation',
    'test_samples': len(all_labels),
    'valid_samples': len(valid_labels),
    'parsing_errors': parsing_errors,
    'evaluation_duration_minutes': eval_duration / 60,
    'metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1)
    },
    'confusion_matrix': cm.tolist(),
    'classification_report': report,
    'per_class_metrics': {
        CLASS_NAMES[i]: {
            'precision': float(precision_per_class[i]),
            'recall': float(recall_per_class[i]),
            'f1_score': float(f1_per_class[i]),
            'support': int(support_per_class[i]),
            'errors': int(errors_by_class[CLASS_NAMES[i]])
        }
        for i in range(len(CLASS_NAMES))
    },
    'comparison': comparison,
    'target_achieved': bool(accuracy >= 0.95)
}

# Sauvegarder en JSON
results_path = OUTPUT_DIR / 'evaluation_results_phase2.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print("="*70)
print("RÉSULTATS SAUVEGARDÉS")
print("="*70)
print(f"✓ Métriques JSON: {results_path}")
print(f"✓ Confusion Matrix: {cm_path}")
print(f"✓ Per-Class Metrics: {metrics_path}")
print("="*70)

print("\n" + "="*70)
print("ÉVALUATION COMPLÈTE TERMINÉE !")
print("="*70)
print(f"Accuracy finale: {accuracy:.1%}")
print(f"F1-Score:        {f1:.4f}")
print(f"Target atteint:  {'✅ OUI' if accuracy >= 0.95 else '⚠️ NON'}")
print("="*70)

## 12. Résumé et Prochaines Étapes

### ✓ Accompli dans ce notebook:
1. **Modèle Phase 2 chargé** - Best checkpoint
2. **Évaluation sur test set** - 3,050 images
3. **Métriques calculées** - Accuracy, F1, Precision, Recall
4. **Confusion matrix** - Analyse détaillée par classe
5. **Comparaison** - Baseline → Phase 1 → Phase 2

### 📊 Résultats:
- **Accuracy**: Voir ci-dessus
- **F1-Score**: Voir ci-dessus
- **Target (95-97%)**: Voir ci-dessus

### 📋 Prochaines étapes:

**Si target atteint (≥95%)**:
1. **Notebook 14**: Export & Quantification
   - Export FP16 (cloud deployment)
   - Quantification INT8 (edge deployment)
   - Quantification INT4 (mobile deployment)
   - Benchmarks de performance

**Si target non atteint (<95%)**:
1. Analyser les erreurs dans la confusion matrix
2. Identifier les classes problématiques
3. Ajuster hyperparamètres Phase 2:
   - Augmenter learning rate
   - Plus d'epochs
   - Ajuster LoRA dropout
4. Relancer Phase 2 training

### 📁 Fichiers générés:
- `outputs/evaluation_results_phase2.json` - Métriques complètes
- `outputs/confusion_matrix_phase2.png` - Visualisation
- `outputs/per_class_metrics_phase2.png` - Métriques par classe